In [ ]:
# Test dataset: questions, contexts, and ground truth answers
# This dataset contains multiple Q&A pairs to evaluate model performance
# Each item has: a question, a context paragraph, and the expected answer
qa_test_data = [
    {
        'question': 'What is the immune system?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a system of many biological structures and processes within an organism that protects against disease'
    },
    {
        'question': 'What must the immune system detect?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a wide variety of agents, known as pathogens'
    },
    {
        'question': 'What are pathogens?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'agents from viruses to parasitic worms'
    }
]

def get_answer(question, context):
    """
    Extract answer from context using BERT Q&A model.
    
    This function:
    1. Prepares the question and context with special tokens
    2. Tokenizes both inputs
    3. Feeds them to the BERT model
    4. Returns the predicted answer span and confidence score
    
    Args:
        question: The question to answer
        context: The paragraph containing the answer
        
    Returns:
        answer: The predicted answer text
        confidence: Combined confidence score (start + end logits)
    """
    # Prepare input: Add [CLS] at start of question, [SEP] at end of both
    question_text = '[CLS] ' + question + '[SEP]'
    context_text = context + '[SEP]'
    
    # Tokenize: Convert text to WordPiece tokens
    question_tokens = tokenizer.tokenize(question_text)
    context_tokens = tokenizer.tokenize(context_text)
    
    # Combine tokens and convert to numerical IDs that BERT understands
    tokens = question_tokens + context_tokens
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    # Segment IDs: 0 for question tokens, 1 for context tokens
    # This helps BERT distinguish between question and context
    segment_ids = [0] * len(question_tokens) + [1] * len(context_tokens)
    
    # Convert to PyTorch tensors (add batch dimension with [])
    input_ids = torch.tensor([input_ids])
    segment_ids = torch.tensor([segment_ids])
    
    # Get model output: BERT predicts start and end positions of the answer
    output = model(input_ids=input_ids, token_type_ids=segment_ids)
    
    # Get indices with highest scores: these mark answer boundaries
    start_index = torch.argmax(output.start_logits)
    end_index = torch.argmax(output.end_logits)
    
    # Extract answer: Join tokens between start and end indices
    answer = ' '.join(tokens[start_index:end_index+1])
    
    # Calculate confidence: Higher scores = more confident prediction
    start_score = torch.max(output.start_logits).item()
    end_score = torch.max(output.end_logits).item()
    confidence = start_score + end_score
    
    return answer, confidence

# Evaluate the Q&A model on test dataset
print("Evaluating Q&A Model:\n")
print("="*80)

# Store results for each test case
results = []

# Iterate through each question-answer pair in test dataset
for i, item in enumerate(qa_test_data, 1):
    # Get model prediction and confidence score
    predicted_answer, confidence = get_answer(item['question'], item['context'])
    
    # Calculate metrics: Exact Match and F1 score
    em = calculate_exact_match(predicted_answer, item['answer'])
    f1 = calculate_f1(predicted_answer, item['answer'])
    
    # Save results for averaging later
    results.append({
        'em': em,
        'f1': f1,
        'confidence': confidence
    })
    
    # Display results for this example
    print(f"\nExample {i}:")
    print(f"Question: {item['question']}")
    print(f"Predicted: {predicted_answer}")
    print(f"Truth: {item['answer']}")
    print(f"EM: {em}  |  F1: {f1:.3f}  |  Confidence: {confidence:.2f}")

print("\n" + "="*80)

# Calculate average metrics across all test examples
# This gives us overall model performance
avg_em = sum(r['em'] for r in results) / len(results)
avg_f1 = sum(r['f1'] for r in results) / len(results)
avg_confidence = sum(r['confidence'] for r in results) / len(results)

print(f"\n📊 Average Performance:")
print(f"  Exact Match: {avg_em:.2%}")
print(f"  F1 Score: {avg_f1:.3f}")
print(f"  Avg Confidence: {avg_confidence:.2f}")

print("\n" + "="*80)

### Evaluate on Multiple Examples

Now let's test our Q&A model on several question-context pairs and calculate average metrics.

**Why evaluate on multiple examples?**
A single example might give a misleading impression of model performance. By testing on multiple diverse questions, we can:

1. **Measure overall accuracy**: Average metrics across examples give us the true model quality
2. **Identify patterns**: See what types of questions the model handles well vs. poorly
3. **Detect overfitting**: Ensure the model generalizes beyond single examples
4. **Build confidence**: Multiple successful predictions indicate reliable performance

**The evaluation process:**
1. Define a test dataset with question-context-answer triples
2. For each example, get the model's prediction
3. Calculate both EM and F1 scores comparing prediction to ground truth
4. Track confidence scores (how certain the model is)
5. Compute aggregate statistics (mean EM, mean F1, etc.)

This systematic evaluation approach is standard practice in NLP research and production Q&A systems.

In [ ]:
def normalize_answer(text):
    """
    Normalize answer text for fair comparison between prediction and ground truth.
    
    Normalization steps:
    - Convert to lowercase (so "System" matches "system")
    - Remove punctuation (so "disease." matches "disease")
    - Remove articles (a, an, the) as they don't affect meaning
    - Remove extra whitespace
    
    This allows us to focus on content similarity rather than formatting differences.
    """
    import re
    import string
    
    # Convert to lowercase for case-insensitive comparison
    text = text.lower()
    
    # Remove all punctuation marks (replace with space to avoid joining words)
    text = ''.join(ch if ch not in string.punctuation else ' ' for ch in text)
    
    # Remove common articles that don't affect answer meaning
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    
    # Collapse multiple spaces into single space and trim
    text = ' '.join(text.split())
    
    return text

def calculate_exact_match(prediction, ground_truth):
    """
    Calculate Exact Match (EM) score.
    
    EM is a strict metric: the prediction must match the ground truth exactly
    after normalization. This is the primary metric for Q&A systems.
    
    Returns:
        1 if normalized strings match exactly
        0 otherwise (no partial credit)
    """
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def calculate_f1(prediction, ground_truth):
    """
    Calculate F1 score based on token-level overlap.
    
    F1 is more lenient than Exact Match - it gives partial credit for
    answers that share some words with the ground truth.
    
    F1 = 2 * (precision * recall) / (precision + recall)
    
    Where:
    - Precision = What fraction of predicted words are correct?
    - Recall = What fraction of ground truth words were predicted?
    
    Example:
    - Prediction: "biological system structures"
    - Truth: "biological structures and processes"
    - Common: {"biological", "structures"}
    - Precision: 2/3, Recall: 2/4, F1: 0.571
    """
    # Tokenize normalized answers into individual words
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    
    # Handle empty predictions or ground truths
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    
    # Find common tokens using set intersection
    common_tokens = set(pred_tokens) & set(truth_tokens)
    
    # If no overlap, F1 is 0
    if len(common_tokens) == 0:
        return 0
    
    # Calculate precision: correct tokens / predicted tokens
    precision = len(common_tokens) / len(pred_tokens)
    
    # Calculate recall: correct tokens / ground truth tokens
    recall = len(common_tokens) / len(truth_tokens)
    
    # Calculate F1 score: harmonic mean of precision and recall
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return f1

# Test the metrics with example cases to understand how they work
print("Testing Metrics:\n")
print("="*80)

# Test cases showing different levels of match quality
test_cases = [
    ("a system of many biological structures", "a system of many biological structures", "Perfect match"),
    ("system of biological structures", "a system of many biological structures", "Missing words"),
    ("the biological system", "a system of many biological structures", "Partial match"),
    ("completely wrong answer", "a system of many biological structures", "No match"),
]

for pred, truth, description in test_cases:
    # Calculate both metrics
    em = calculate_exact_match(pred, truth)
    f1 = calculate_f1(pred, truth)
    
    print(f"\n{description}:")
    print(f"  Predicted: '{pred}'")
    print(f"  Truth: '{truth}'")
    print(f"  EM: {em}  |  F1: {f1:.3f}")

print("\n" + "="*80)

# Q&A with finetuned BERT

In this section, we'll explore how to perform **extractive question answering** using a pre-trained BERT model that has been fine-tuned on the SQuAD (Stanford Question Answering Dataset). 

**Extractive Q&A** means the model finds and extracts the answer directly from the given context paragraph - it doesn't generate new text, but rather identifies the start and end positions of the answer span within the existing text.

This approach is particularly useful for:
- Reading comprehension tasks
- Information extraction from documents
- Building intelligent search and retrieval systems
- Creating FAQ systems that can find relevant answers in documentation

First, let us import the necessary modules:

In [23]:
%%capture
!pip install transformers==3.5.1

In [ ]:
# Import necessary libraries
import torch  # PyTorch for tensor operations
from transformers import BertForQuestionAnswering, BertTokenizer  # Hugging Face BERT models


Now, we download and load the model. We use the **bert-large-uncased-whole-word-masking-finetuned-squad** model which is finetuned on the SQUAD (Stanford Question Answering Dataset).

This specific model has several important characteristics:
- **bert-large**: Uses the larger BERT architecture with 24 transformer layers (340M parameters) instead of the base version's 12 layers. This provides better accuracy at the cost of slower inference.
- **uncased**: The model treats all text as lowercase, meaning "Hello" and "hello" are treated identically. This reduces vocabulary size and works well for most English Q&A tasks.
- **whole-word-masking**: During pre-training, entire words were masked together rather than individual tokens, leading to better language understanding.
- **finetuned-squad**: The model has been specifically trained on the SQuAD dataset, which contains 100,000+ question-answer pairs from Wikipedia articles, making it excellent for extractive question answering tasks.


In [ ]:
# Load pre-trained BERT model fine-tuned on SQuAD dataset
# This model is specifically trained for extractive question answering
# - bert-large: Larger model with 24 layers (better accuracy, slower)
# - uncased: Not case-sensitive (treats "Hello" and "hello" the same)
# - whole-word-masking: Better pre-training technique
# - finetuned-squad: Trained on Stanford Question Answering Dataset
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


Next, we download and load the tokenizer which is used for pre-training the bert-large-uncased-whole-word-masking-finetuned-squad model.

**Why we need the tokenizer:**
The tokenizer is crucial because it transforms raw text into the exact format that BERT expects. It must match the model's training process, including:
- The same vocabulary (30,000+ WordPiece tokens)
- The same special tokens ([CLS], [SEP], [PAD], etc.)
- The same tokenization rules (how words are split into subwords)

Using a mismatched tokenizer would result in poor performance or errors, as the model wouldn't recognize the input tokens correctly.


In [ ]:
# Load the tokenizer that matches the pre-trained model
# The tokenizer converts text to tokens (subwords) that BERT can understand
# It must match the model's vocabulary and special tokens
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


Now that we downloaded the model and tokenizer, let's preprocess the input.

## Preprocessing the input

Preprocessing is a critical step in using BERT for question answering. BERT expects inputs in a very specific format:

1. **Special tokens**: [CLS] at the beginning and [SEP] tokens to separate segments
2. **Token sequence**: Question tokens followed by context tokens
3. **Segment IDs**: Binary values (0 or 1) indicating which segment each token belongs to
4. **Input IDs**: Numerical representations of each token

This structured format allows BERT to:
- Understand which part is the question vs. the context
- Process both pieces of information together
- Identify answer boundaries within the context

First, we define the input to the BERT which is the question and paragraph text:

In [ ]:
# Define the question we want to answer
question = "What is the immune system?"

# Define the context paragraph that contains the answer
# BERT will extract a span from this paragraph as the answer
paragraph = "The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism's own healthy tissue."

**Add special tokens**: Add [CLS] token to the beginning of the question and [SEP] token at the end of both the question and paragraph.

These special tokens serve specific purposes in BERT's architecture:
- **[CLS]** (Classification): A special token that goes at the beginning of every input. Its final hidden state is often used for classification tasks, though for Q&A we focus on individual token positions.
- **[SEP]** (Separator): Marks boundaries between different segments of text. It helps BERT understand where one piece of text ends and another begins.

The resulting format looks like:
```
[CLS] What is the immune system? [SEP] The immune system is... [SEP]
```

This structure is essential for BERT to properly process the question-context pair.

In [ ]:
# Add special BERT tokens:
# [CLS] = Classification token at the start (required by BERT)
# [SEP] = Separator token to mark end of segments
question = '[CLS] ' + question + '[SEP]'
paragraph = paragraph + '[SEP]'


Now, tokenize the question and paragraph.

**Tokenization** converts text into subword units called **WordPiece tokens**. This approach handles:
- **Rare words**: Unknown words are broken into known subwords
- **Vocabulary efficiency**: A fixed vocabulary of ~30K tokens can represent virtually any word
- **Morphology**: Related words share subword components (e.g., "play", "playing", "played" share "play")

Example tokenization:
- "immune" → ["im", "##mune"]
- "system" → ["system"]
- "pathogens" → ["path", "##ogen", "##s"]

The "##" prefix indicates a token that continues from the previous one (not a word beginning).


In [ ]:
# Tokenize: Convert text into WordPiece tokens
# Example: "immune" might become ["im", "##mune"]
# The ## prefix indicates a continuation of the previous word
question_tokens = tokenizer.tokenize(question)
paragraph_tokens = tokenizer.tokenize(paragraph)



**Combine and convert**: Combine the question and paragraph tokens and convert them to input_ids.

This step creates the final input sequence that BERT will process:
1. **Concatenation**: Join question tokens and paragraph tokens into one sequence
2. **Vocabulary mapping**: Convert each token string to its corresponding integer ID in BERT's vocabulary

For example:
- Token "[CLS]" → ID 101
- Token "what" → ID 2054
- Token "##mune" → ID 18900

These numerical IDs are what the model actually processes - BERT's embedding layer will convert them to high-dimensional vectors.

In [ ]:
# Combine question and paragraph tokens into single sequence
# BERT processes both together: [CLS] question [SEP] paragraph [SEP]
tokens = question_tokens + paragraph_tokens

# Convert tokens to numerical IDs using the vocabulary
# Each token maps to a unique integer ID (e.g., "the" -> 1996)
input_ids = tokenizer.convert_tokens_to_ids(tokens)



Next, we define the **segment_ids** (also called token_type_ids). The segment_ids will be 0 for all the tokens of the question and it will be 1 for all the tokens of the paragraph.

**Why segment IDs matter:**
BERT was pre-trained on sentence pair tasks, so it learned to distinguish between different segments using these IDs. For question answering:
- **Segment 0** (question): Tells BERT "this is what we're asking about"
- **Segment 1** (context): Tells BERT "search for the answer in this part"

This binary signal helps BERT:
- Apply different attention patterns to questions vs. contexts
- Understand the relationship between the question and potential answer spans
- Improve accuracy by leveraging segment-specific learned representations

Example:
```
Tokens:      [CLS] What is... [SEP] The immune system... [SEP]
Segment IDs:   0    0  0      0      1    1      1          1
```


In [ ]:
# Create segment IDs (also called token_type_ids)
# These tell BERT which tokens belong to the question (0) vs. paragraph (1)
# Example: [0, 0, 0, 0, 1, 1, 1, 1, 1, ...]
#          [CLS] What is ... [SEP] The immune system ...
segment_ids = [0] * len(question_tokens)  # All question tokens get 0
segment_ids += [1] * len(paragraph_tokens)  # All paragraph tokens get 1


Now we convert the input_ids and segment_ids to PyTorch tensors.

**Why tensors:**
PyTorch (and deep learning models in general) operate on tensors - multi-dimensional arrays optimized for:
- **GPU acceleration**: Tensors can be efficiently processed on GPUs for faster inference
- **Batch processing**: The extra dimension allows processing multiple examples simultaneously
- **Automatic differentiation**: Enables gradient computation (useful during training)

The `[input_ids]` notation adds a **batch dimension**:
- Original shape: (sequence_length,)
- Tensor shape: (1, sequence_length)

The batch size of 1 means we're processing a single question-context pair. In production systems, you'd typically batch multiple examples together for efficiency.

In [ ]:
# Convert to PyTorch tensors
# The [] adds a batch dimension (batch_size=1) since we're processing one example
# Shape: (1, sequence_length)
input_ids = torch.tensor([input_ids])
segment_ids = torch.tensor([segment_ids])



Now that we processed the input, let's feed them to the model and get the result.

## Getting the answer

We feed the input_ids and segment_ids to the model which returns the **start score** and **end score** for all of the tokens.

**How BERT Q&A works:**
BERT doesn't directly predict the answer text. Instead, it predicts:
1. **Start logits**: A score for each token position indicating how likely that position is the **start** of the answer
2. **End logits**: A score for each token position indicating how likely that position is the **end** of the answer

The model processes the entire sequence through 24 transformer layers, building rich contextual representations. Then:
- A linear layer computes start scores from each token's representation
- Another linear layer computes end scores

We then select the token positions with the highest scores as our answer boundaries. The tokens between (and including) these positions form the extracted answer.

This approach ensures the answer is always a continuous span from the original context - pure extractive Q&A.

In [ ]:
# Feed inputs to the BERT Q&A model
# Model outputs:
# - start_logits: scores for each token being the START of the answer
# - end_logits: scores for each token being the END of the answer
output = model(input_ids=input_ids, token_type_ids = segment_ids)

In [ ]:
# Extract the start and end scores (logits) for each token position
# These are raw scores before softmax - higher score = more likely to be answer boundary
start_scores = output.start_logits
end_scores = output.end_logits


Now, we select the **start_index** which is the index of the token which has a maximum start score and **end_index** which is the index of the token which has a maximum end score.

**Finding the answer span:**
- **argmax()** finds the position with the highest score
- The start_index points to where the answer begins in the token sequence
- The end_index points to where the answer ends

**Important notes:**
- In a simple implementation like this, we take the single highest scores independently
- More sophisticated approaches might:
  - Ensure end_index ≥ start_index
  - Consider multiple candidate spans
  - Use score thresholds to detect unanswerable questions
  - Limit answer length to reasonable spans

For most cases, this simple approach works well because the model has learned to predict sensible answer boundaries during fine-tuning on SQuAD.

In [ ]:
# Find the token indices with the highest scores
# argmax returns the position of the maximum value
# start_index: where the answer begins
# end_index: where the answer ends
start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)


That's it! Now, we print the text span between the start and end index as our answer.

**Reconstructing the answer:**
We extract all tokens from start_index to end_index (inclusive) and join them with spaces to reconstruct the answer text.

Note that because we used WordPiece tokenization, the answer may contain tokens like "##mune" which need to be reassembled. When printed, they appear with spaces, but in production you'd typically:
- Remove "##" prefixes
- Join subwords without spaces
- Apply text post-processing for better readability

The result is a span of text extracted directly from the original context paragraph - this is the model's answer to our question!

In [ ]:
# Extract and print the answer: join all tokens between start and end indices
# +1 because Python slicing is exclusive at the end
# This reconstructs the answer text from the token sequence
print(' '.join(tokens[start_index:end_index+1]))

---

## Evaluating Q&A Performance with Metrics

The example above shows how to get a single answer, but **how do we measure if our model is performing well?** 

In machine learning, especially for Q&A systems, we need objective metrics to:
- **Benchmark performance**: Compare different models or approaches
- **Track improvements**: Measure if changes actually help
- **Identify weaknesses**: Find what types of questions the model struggles with
- **Report results**: Communicate model quality to stakeholders

Simply getting an answer isn't enough - we need to know if it's the *right* answer and how close it is to what we expect.

Let's look at key metrics for Question Answering systems and implement them to evaluate our BERT model.

### Understanding Q&A Metrics

For extractive Question Answering (like BERT Q&A), we use two main metrics that originated from the SQuAD benchmark:

**1. Exact Match (EM)**
- The predicted answer must match the ground truth exactly (after normalization)
- **Binary**: either 1 (perfect match) or 0 (no match)
- **Strict** but clear measure of success
- Pros: Unambiguous, easy to interpret
- Cons: No partial credit - "biological system" vs "biological structures" both score 0

**2. F1 Score**
- Token-level overlap between prediction and ground truth
- **Range**: 0.0 (no overlap) to 1.0 (perfect match)
- Measures partial credit for partially correct answers
- **More lenient** than EM - recognizes when a model is "close"
- Balances precision (accuracy of predicted tokens) and recall (coverage of ground truth tokens)

**Why both metrics?**
- EM tells us **complete accuracy** - critical for production systems
- F1 tells us **how close we are** - useful for model development and understanding mistakes
- Together, they give a complete picture of model performance

**Standard practice:** Report both metrics, with EM often being the primary metric since users typically want fully correct answers.

Let's implement these metrics and test them on multiple examples.